# R3-N2 Bi-GRU candidate-bundle latency benchmark

Benchmarks the R3-N2 Bi-GRU candidate bundle on the current Colab CPU using the production inference pipeline. This notebook does not fit or alter the model, calibration, threshold, or bundle. Fresh subprocess startup is reported separately; it is not a cold VM measurement.

Select already-approved recordings in the fixture cell. The candidate's calibration review status is recorded; the explicit pending-review override is used only to run this validation benchmark and does not approve the bundle.

After each Colab runtime restart, execute the next code cell first to capture the separate fresh-runtime cold-start measurement. It mounts Drive itself. Continue with the original notebook cells only after its report is saved.


In [22]:
# Fresh-Runtime Cold-Start Benchmark
# Run this cell first after restarting the Colab runtime.
# Exactly one production prediction. Zero warm-up calls.

import time
COLD_START_NS = time.perf_counter_ns()

if globals().get("_COLD_START_ATTEMPTED", False):
    raise RuntimeError("Restart the Colab runtime before taking another cold-start measurement.")
_COLD_START_ATTEMPTED = True

COLD_ERRORS = []
COLD_PHASE = "initialization"
ROOT = None
COLD_BUNDLE = None
COLD_FIXTURE = None
COLD_DURATION = None
COLD_RESULT = None
COLD_DEVICE = None
COLD_IDENTITY = None
COLD_REVIEW = None

COLD_MODEL_INIT_NS = None
COLD_BUNDLE_LOAD_NS = None
COLD_PREDICTION_NS = None
COLD_TOTAL_NS = None
COLD_FAILURE_ELAPSED_NS = None
DRIVE_MOUNT_NS = None
PATH_RESOLUTION_NS = None
FIXTURE_PREPARATION_NS = None
BUNDLE_IDENTITY_CAPTURE_NS = None

COLD_PREDICTION_CALLS = 0
COLD_BUNDLE_LOAD_SUCCEEDED = False
COLD_CREATED_UTC = None

try:
    import json
    import os
    import sys
    import platform
    import hashlib
    import math
    import importlib.metadata as metadata
    from pathlib import Path
    from datetime import datetime, timezone

    COLD_CREATED_UTC = datetime.now(timezone.utc).isoformat()

    COLD_PHASE = "Google Drive mount"
    mount_ns = time.perf_counter_ns()
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
    finally:
        DRIVE_MOUNT_NS = time.perf_counter_ns() - mount_ns

    COLD_PHASE = "project path resolution"
    path_ns = time.perf_counter_ns()
    try:
        ROOT_CANDIDATES = [
            Path("/content/drive/MyDrive/PAI&AIES Project"),
            Path("/content/drive/MyDrive/EchoProof"),
        ]
        ROOT = next(
            (p for p in ROOT_CANDIDATES if (p / "src/echoproof/inference.py").is_file()),
            None,
        )
        if ROOT is None:
            raise FileNotFoundError(
                "EchoProof root not found. Check ROOT_CANDIDATES."
            )

        os.chdir(ROOT)
        COLD_BUNDLE = (
            ROOT / "runs/r3/r3_n1_bigru/selected_bigru_v1/handoff/bundle"
        )
        if not (COLD_BUNDLE / "bundle.json").is_file():
            raise FileNotFoundError(f"Bi-GRU bundle missing: {COLD_BUNDLE}")

        COLD_REVIEW = (
            json.loads((COLD_BUNDLE / "bundle.json").read_text(encoding="utf-8"))
            .get("extra", {})
            .get("review_status", "unrecorded")
        )
    finally:
        PATH_RESOLUTION_NS = time.perf_counter_ns() - path_ns

    # Same fixture paths and duration choices as the regular benchmark.
    COLD_START_FIXTURES = {
        1: ROOT / "dataset/genuine/wav/19-198-0000.wav",
        4: ROOT / "dataset/genuine/wav/19-198-0037.wav",
        6: ROOT / "dataset/genuine/wav/89-218-0004.wav",
        10: ROOT / "dataset/genuine/wav/19-198-0002.wav",
        15: ROOT / "dataset/genuine/wav/911-128684-0002.wav",
    }

    COLD_START_DURATION_SECONDS = 1
    if COLD_START_DURATION_SECONDS not in COLD_START_FIXTURES:
        raise ValueError("Choose an existing COLD_START_FIXTURES key.")

    COLD_DURATION = COLD_START_DURATION_SECONDS
    COLD_FIXTURE = Path(COLD_START_FIXTURES[COLD_DURATION])

    COLD_PHASE = "model/library initialization"
    model_init_ns = time.perf_counter_ns()
    try:
        import numpy
        import scipy
        import soundfile as sf
        import sklearn
        import joblib
        import librosa
        import torch

        from echoproof.inference import analyze
        from echoproof.models.r3_n1_bigru_bundle import (
            load_calibrated_bigru_bundle,
        )

        torch.set_num_threads(1)
    finally:
        COLD_MODEL_INIT_NS = time.perf_counter_ns() - model_init_ns

    COLD_PHASE = "fixture preparation"
    fixture_ns = time.perf_counter_ns()
    try:
        import io

        if not COLD_FIXTURE.is_file():
            raise FileNotFoundError(f"Fixture missing: {COLD_FIXTURE}")

        with sf.SoundFile(io.BytesIO(COLD_FIXTURE.read_bytes())) as audio:
            required_frames = int(audio.samplerate * COLD_DURATION)
            if required_frames > audio.frames:
                raise ValueError(
                    f"Fixture is shorter than {COLD_DURATION} seconds."
                )

            samples = audio.read(
                frames=required_frames,
                dtype="float32",
                always_2d=True,
            )
            rate = audio.samplerate

        audio_buffer = io.BytesIO()
        sf.write(
            audio_buffer,
            samples,
            rate,
            format="WAV",
            subtype="PCM_16",
        )
        audio_bytes = audio_buffer.getvalue()
    finally:
        FIXTURE_PREPARATION_NS = time.perf_counter_ns() - fixture_ns

    COLD_PHASE = "bundle loading"
    load_ns = time.perf_counter_ns()
    try:
        loaded = load_calibrated_bigru_bundle(
            COLD_BUNDLE,
            device="cpu",
            batch_size=1,
            allow_pending_review=True,
        )

        if loaded.spec.model_version != "bigru:B":
            raise ValueError(
                f"Unexpected model version: {loaded.spec.model_version}"
            )

        COLD_BUNDLE_LOAD_SUCCEEDED = True
    finally:
        COLD_BUNDLE_LOAD_NS = time.perf_counter_ns() - load_ns

    COLD_PHASE = "first audio prediction"
    COLD_PREDICTION_CALLS = 1
    prediction_ns = time.perf_counter_ns()
    try:
        result = analyze(audio_bytes, loaded).result
    finally:
        COLD_PREDICTION_NS = time.perf_counter_ns() - prediction_ns
        COLD_TOTAL_NS = time.perf_counter_ns() - COLD_START_NS

    COLD_RESULT = {
        "status": result.status,
        "score": result.score,
        "assessment": result.assessment,
        "model_version": result.model_version,
        "valid": bool(
            result.status == "success"
            and result.score is not None
            and math.isfinite(float(result.score))
            and 0 <= float(result.score) <= 1
            and result.assessment in ("genuine", "synthetic")
            and result.model_version == "bigru:B"
        ),
    }

    if not COLD_RESULT["valid"]:
        COLD_ERRORS.append({
            "phase": COLD_PHASE,
            "type": "InvalidPrediction",
            "message": "The single production prediction failed validation.",
        })

except Exception as exc:
    COLD_FAILURE_ELAPSED_NS = time.perf_counter_ns() - COLD_START_NS
    COLD_ERRORS.append({
        "phase": COLD_PHASE,
        "type": type(exc).__name__,
        "message": str(exc),
    })
    print(
        f"Cold-start measurement failed during {COLD_PHASE}: "
        f"{type(exc).__name__}: {exc}",
        flush=True,
    )


# Collect environment and bundle identity AFTER the prediction.
# Hashing is deliberately excluded from the prediction timing.
COLD_REPORT_ROOT = ROOT

if COLD_REPORT_ROOT is not None:
    try:
        packages = {}
        for package_name in [
            "numpy", "scipy", "soundfile", "scikit-learn",
            "joblib", "librosa", "torch", "echoproof",
        ]:
            try:
                packages[package_name] = metadata.version(package_name)
            except metadata.PackageNotFoundError:
                packages[package_name] = None

        COLD_DEVICE = {
            "platform": platform.platform(),
            "machine": platform.machine(),
            "processor": platform.processor(),
            "cpu_count_logical": os.cpu_count(),
            "python": sys.version,
            "packages": packages,
            "device": "CPU",
            "torch_device": "cpu",
            "torch_num_threads": (
                torch.get_num_threads() if "torch" in locals() else None
            ),
            "cuda_available": (
                torch.cuda.is_available() if "torch" in locals() else None
            ),
            "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        }

        try:
            with open("/proc/cpuinfo", encoding="utf-8") as f:
                COLD_DEVICE["cpuinfo_model_name"] = next(
                    (
                        line.split(":", 1)[1].strip()
                        for line in f
                        if line.lower().startswith("model name")
                    ),
                    None,
                )
        except OSError:
            COLD_DEVICE["cpuinfo_model_name"] = None

        try:
            COLD_DEVICE["colab_release_file"] = Path(
                "/etc/colab-release"
            ).read_text(errors="replace").strip()
        except OSError:
            COLD_DEVICE["colab_release_file"] = None

        if COLD_BUNDLE is not None and COLD_BUNDLE.is_dir():
            hash_ns = time.perf_counter_ns()

            def sha256_file(path):
                digest = hashlib.sha256()
                with open(path, "rb") as file:
                    for chunk in iter(lambda: file.read(1024 * 1024), b""):
                        digest.update(chunk)
                return digest.hexdigest()

            bundle_files = sorted(
                p for p in COLD_BUNDLE.rglob("*") if p.is_file()
            )

            COLD_IDENTITY = {
                "bundle_path": str(COLD_BUNDLE),
                "bundle_json_sha256": sha256_file(COLD_BUNDLE / "bundle.json"),
                "files": {
                    p.relative_to(COLD_BUNDLE).as_posix(): sha256_file(p)
                    for p in bundle_files
                },
            }

            BUNDLE_IDENTITY_CAPTURE_NS = (
                time.perf_counter_ns() - hash_ns
            )

    except Exception as exc:
        COLD_ERRORS.append({
            "phase": "post-measurement metadata",
            "type": type(exc).__name__,
            "message": str(exc),
        })

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    report_path = (
        COLD_REPORT_ROOT
        / "reports/r3"
        / f"r3_n2_bigru_fresh_runtime_cold_start_{stamp}.json"
    )

    report = {
        "schema": "r3-n2-bigru-fresh-runtime-cold-start-v1",
        "model": "Bi-GRU candidate bundle",
        "benchmark_type": (
            "fresh-runtime cold-start; single first prediction; not cold VM start"
        ),
        "created_utc": COLD_CREATED_UTC,
        "project_root": str(ROOT),
        "timing_scope": {
            "clock": "time.perf_counter_ns",
            "prediction_calls": COLD_PREDICTION_CALLS,
            "warmup_calls": 0,
            "measurement_start": (
                "immediately after importing time at the start of this cell"
            ),
            "measurement_end": (
                "immediately after the first analyze() call returns or raises"
            ),
            "vm_provisioning_time_included": False,
            "note": (
                "Includes Drive mount, project path resolution, fixture preparation, "
                "library initialization, bundle loading and first prediction. "
                "Colab VM provisioning is excluded because the runtime is already "
                "available when notebook execution begins."
            ),
        },
        "device": COLD_DEVICE,
        "bundle": COLD_IDENTITY,
        "bundle_review_status": COLD_REVIEW,
        "bundle_load_succeeded": COLD_BUNDLE_LOAD_SUCCEEDED,
        "fixture": {
            "path": str(COLD_FIXTURE) if COLD_FIXTURE else None,
            "duration_seconds": COLD_DURATION,
        },
        "measurements_ns": {
            "cell_start_to_first_prediction_total": COLD_TOTAL_NS,
            "model_library_initialization": COLD_MODEL_INIT_NS,
            "bundle_loading": COLD_BUNDLE_LOAD_NS,
            "first_audio_prediction": COLD_PREDICTION_NS,
            "cell_start_to_failure_elapsed": COLD_FAILURE_ELAPSED_NS,
            "drive_mount": DRIVE_MOUNT_NS,
            "project_path_resolution": PATH_RESOLUTION_NS,
            "fixture_preparation": FIXTURE_PREPARATION_NS,
            "post_measurement_bundle_identity_capture": BUNDLE_IDENTITY_CAPTURE_NS,
        },
        "prediction": COLD_RESULT,
        "errors": COLD_ERRORS,
    }

    required_timing_fields = {
        "cell_start_to_first_prediction_total",
        "model_library_initialization",
        "bundle_loading",
        "first_audio_prediction",
        "cell_start_to_failure_elapsed",
        "drive_mount",
        "project_path_resolution",
        "fixture_preparation",
        "post_measurement_bundle_identity_capture",
    }

    timing_values = report["measurements_ns"]
    schema_valid = (
        report["schema"] == "r3-n2-bigru-fresh-runtime-cold-start-v1"
        and report["benchmark_type"].startswith("fresh-runtime cold-start")
    )
    timing_fields_valid = (
        set(timing_values) == required_timing_fields
        and all(
            value is None or (type(value) is int and value >= 0)
            for value in timing_values.values()
        )
    )
    call_contract_valid = (
        report["timing_scope"]["prediction_calls"] in (0, 1)
        and report["timing_scope"]["warmup_calls"] == 0
    )

    report["report_validation"] = {
        "schema_valid": schema_valid,
        "timing_fields_valid": timing_fields_valid,
        "at_most_one_prediction_zero_warmup_valid": call_contract_valid,
        "passed": bool(
            schema_valid and timing_fields_valid and call_contract_valid
        ),
    }

    if not report["report_validation"]["passed"]:
        report["errors"].append({
            "phase": "report validation",
            "type": "InvalidReportSchema",
            "message": (
                "Cold-start report schema, timing fields, or "
                "one-call/no-warmup contract failed validation."
            ),
        })

    try:
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(
            json.dumps(report, indent=2, ensure_ascii=False, default=str) + "\n",
            encoding="utf-8",
        )

        print("Fresh-runtime cold-start report:", report_path)
        print("Prediction valid:", (
            None if COLD_RESULT is None else COLD_RESULT["valid"]
        ))
        print("Bundle load succeeded:", COLD_BUNDLE_LOAD_SUCCEEDED)
        print("Report validation:", report["report_validation"]["passed"])
        print(json.dumps(report["measurements_ns"], indent=2))

    except Exception as exc:
        print(
            f"Could not save cold-start report: "
            f"{type(exc).__name__}: {exc}",
            flush=True,
        )

else:
    print("Report not saved: EchoProof project root could not be resolved.")

Fresh-runtime cold-start report: /content/drive/MyDrive/PAI&AIES Project/reports/r3/r3_n2_bigru_fresh_runtime_cold_start_20261114T121037Z.json
Prediction valid: True
Bundle load succeeded: True
Report validation: True
{
  "cell_start_to_first_prediction_total": 1328453227,
  "model_library_initialization": 1073276605,
  "bundle_loading": 213159331,
  "first_audio_prediction": 24022157,
  "cell_start_to_failure_elapsed": null,
  "drive_mount": 750210,
  "project_path_resolution": 6068260,
  "fixture_preparation": 6280849,
  "post_measurement_bundle_identity_capture": 42920737
}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
CANDIDATE_ROOTS = [Path('/content/drive/MyDrive/PAI&AIES Project')]
ROOT = next((p for p in CANDIDATE_ROOTS if (p / 'src/echoproof/inference.py').is_file()), None)
if ROOT is None: raise FileNotFoundError('EchoProof root not found; edit CANDIDATE_ROOTS to your Drive project folder')
os.chdir(ROOT)
BUNDLE = ROOT / 'runs/r3/r3_n1_bigru/selected_bigru_v1/handoff/bundle'
if not (BUNDLE / 'bundle.json').is_file(): raise FileNotFoundError(f'Candidate bundle missing: {BUNDLE}')
print('Project root:', ROOT)
print('Candidate bundle:', BUNDLE)

## Editable fixture selection

Add explicit paths to existing recordings already approved for this benchmark. Each path must supply the target duration; the benchmark uses the first N seconds in memory and never writes a crop. Missing groups are recorded as missing, never silently omitted. No directory scan occurs.

In [ ]:
# EDIT THESE five entries to point at existing approved recordings.
FIXTURES = {
    1:  ROOT / 'dataset/genuine/wav/19-198-0000.wav',
    4:  ROOT / 'dataset/genuine/wav/19-198-0037.wav',
    6: ROOT / 'dataset/genuine/wav/89-218-0004.wav',
    10: ROOT / 'dataset/genuine/wav/19-198-0002.wav',
    15: ROOT / 'dataset/genuine/wav/911-128684-0002.wav',
}
WARMUP_CALLS = 5
MEASURED_CALLS = 100
TOLERANCE = 1e-10
# Paths can be WAV/FLAC supported by production decode. Provide a custom explicit path per duration.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))

import sys, platform, subprocess, json, time, hashlib, importlib.metadata as metadata
import numpy as np
import soundfile as sf
from datetime import datetime, timezone
from echoproof.models.r3_n1_bigru_bundle import load_calibrated_bigru_bundle
from echoproof.inference import analyze
from echoproof.artifacts import file_digest
from echoproof.models.bigru_adapter import BiGRUBundleAdapter
import echoproof.inference as inference
import echoproof.models.bigru_waveform as bigru_waveform
import echoproof.data.chunks as chunk_module
import torch

def package_versions():
    names = ['numpy','scipy','soundfile','scikit-learn','joblib','librosa','torch','echoproof']
    out = {}
    for name in names:
        try: out[name] = metadata.version(name)
        except metadata.PackageNotFoundError: out[name] = None
    return out

def device_details():
    details = {'platform': platform.platform(), 'machine': platform.machine(), 'processor': platform.processor(), 'cpu_count_logical': os.cpu_count(), 'python': sys.version, 'packages': package_versions(), 'device': 'CPU (BiGRUBundleAdapter PyTorch inference)', 'torch_device': 'cpu', 'torch_num_threads': torch.get_num_threads(), 'cuda_available': torch.cuda.is_available(), 'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES')}
    try:
        with open('/proc/cpuinfo') as f: details['cpuinfo_model_name'] = next((line.split(':',1)[1].strip() for line in f if line.lower().startswith('model name')), None)
    except OSError: details['cpuinfo_model_name'] = None
    try: details['colab_release_file'] = Path('/etc/colab-release').read_text(errors='replace').strip()
    except OSError: details['colab_release_file'] = None
    return details

def percentile(values, q): return float(np.percentile(np.asarray(values, dtype=np.float64), q)) if values else None
def digest_bundle():
    paths = sorted(p for p in BUNDLE.rglob('*') if p.is_file())
    return {'bundle_path': str(BUNDLE), 'bundle_json_sha256': file_digest(BUNDLE/'bundle.json'), 'files': {p.relative_to(BUNDLE).as_posix(): file_digest(p) for p in paths}}

bundle_identity = digest_bundle()
BUNDLE_RECORD = json.loads((BUNDLE / 'bundle.json').read_text(encoding='utf-8'))
BUNDLE_REVIEW_STATUS = BUNDLE_RECORD.get('extra', {}).get('review_status', 'unrecorded')
BUNDLE_DEPLOYMENT_READY = BUNDLE_RECORD.get('extra', {}).get('deployment_ready', False)
loaded = load_calibrated_bigru_bundle(BUNDLE, device='cpu', batch_size=1, allow_pending_review=True)
assert loaded.spec.model_version == 'bigru:B'
assert loaded.score_type == 'calibrated_probability'
bundle_identity['review_status'] = BUNDLE_REVIEW_STATUS
bundle_identity['deployment_ready'] = BUNDLE_DEPLOYMENT_READY
print('Bundle identity captured; candidate production loader succeeded for validation.')
print('Candidate review status:', BUNDLE_REVIEW_STATUS, '| deployment_ready:', BUNDLE_DEPLOYMENT_READY)
print(json.dumps(device_details(), indent=2))

## Benchmark and parity

Normal production `analyze()` is the reference. Stage instrumentation wraps the production decode/window, aggregation/calibration/decision, and Bi-GRU feature/prediction callables temporarily and restores each exact original object in `finally`. Feature extraction and recurrent prediction are timed independently inside the actual production model call path. Total and raw per-call nanosecond samples are preserved.


In [ ]:
def instrumented_analyze(audio_bytes, bundle):
    # Production chain: analyze -> BiGRUBundleAdapter.score_chunks -> WaveformBiGRU.forward
    # -> WaveformBiGRU.features and BiGRU.forward. These wrappers time the actual calls.
    stages = {k: 0 for k in ['decode','chunk_construction','feature_extraction','bigru_prediction','aggregation','calibration','threshold_decision','total']}
    calls = {k: 0 for k in stages}
    original_spec = bundle.spec
    original_decision_rule = original_spec.decision_rule
    adapter = bundle.scorer.__self__
    if not isinstance(adapter, BiGRUBundleAdapter):
        raise TypeError('production scorer is not a BiGRUBundleAdapter')
    model = adapter._model
    originals = {
        'decode_audio': inference.decode_audio,
        'eval_windows': inference.eval_windows,
        'scorer': bundle.scorer,
        'aggregate': bundle.aggregate,
        'calibrate': bundle.calibrate,
        'spec': original_spec,
        'decision_rule': original_decision_rule,
        'model_features': model.features,
        'bigru_forward': model.bigru.forward,
    }
    def timed(key, fn):
        def call(*args, **kwargs):
            start_ns = time.perf_counter_ns()
            calls[key] += 1
            try:
                return fn(*args, **kwargs)
            finally:
                stages[key] += time.perf_counter_ns() - start_ns
        return call
    try:
        inference.decode_audio = timed('decode', originals['decode_audio'])
        inference.eval_windows = timed('chunk_construction', originals['eval_windows'])
        object.__setattr__(model, 'features', timed('feature_extraction', originals['model_features']))
        object.__setattr__(model.bigru, 'forward', timed('bigru_prediction', originals['bigru_forward']))
        object.__setattr__(bundle, 'aggregate', timed('aggregation', originals['aggregate']))
        if originals['calibrate'] is not None:
            object.__setattr__(bundle, 'calibrate', timed('calibration', originals['calibrate']))
        class DecisionProxy:
            def decide(self, score):
                return timed('threshold_decision', originals['decision_rule'].decide)(score)
        object.__setattr__(original_spec, 'decision_rule', DecisionProxy())
        start_ns = time.perf_counter_ns()
        result = analyze(audio_bytes, bundle)
        stages['total'] = time.perf_counter_ns() - start_ns
        calls['total'] += 1
        if calls['feature_extraction'] == 0 or calls['bigru_prediction'] == 0:
            raise RuntimeError('production model did not invoke feature extraction and Bi-GRU prediction wrappers')
        return result, stages, calls
    finally:
        inference.decode_audio = originals['decode_audio']
        inference.eval_windows = originals['eval_windows']
        object.__setattr__(model, 'features', originals['model_features'])
        object.__setattr__(model.bigru, 'forward', originals['bigru_forward'])
        object.__setattr__(bundle, 'aggregate', originals['aggregate'])
        object.__setattr__(bundle, 'calibrate', originals['calibrate'])
        object.__setattr__(original_spec, 'decision_rule', originals['decision_rule'])


def prefix_bytes(path, seconds):
    # In-memory prefix for one explicit fixture; no crop file is written.
    import soundfile as sf, io
    data = Path(path).read_bytes()
    with sf.SoundFile(io.BytesIO(data)) as f:
        if f.samplerate * seconds > f.frames:
            raise ValueError(f'fixture is shorter than {seconds}s')
        samples = f.read(frames=int(f.samplerate*seconds), dtype='float32', always_2d=True)
        rate = f.samplerate
    buf = io.BytesIO()
    sf.write(buf, samples, rate, format='WAV', subtype='PCM_16')
    return buf.getvalue()


def fresh_process(path, duration):
    # Each call launches an independent interpreter. The child measures bundle loading
    # and its first production prediction separately; the parent measures spawn-to-result.
    child = 'import json,sys,time\nfrom pathlib import Path\nsys.path.insert(0, str(Path(sys.argv[1]) / "src"))\nfrom echoproof.models.r3_n1_bigru_bundle import load_calibrated_bigru_bundle\nfrom echoproof.inference import analyze\nimport soundfile as sf,io\nroot,bundle,audio_path,dur=sys.argv[1],sys.argv[2],sys.argv[3],int(sys.argv[4])\ndata=Path(audio_path).read_bytes()\nwith sf.SoundFile(io.BytesIO(data)) as f: x=f.read(frames=int(f.samplerate*dur),dtype="float32",always_2d=True); rate=f.samplerate\nb=io.BytesIO(); sf.write(b,x,rate,format="WAV",subtype="PCM_16"); raw=b.getvalue()\nt=time.perf_counter_ns(); model=load_calibrated_bigru_bundle(bundle,device="cpu",batch_size=1,allow_pending_review=True); load_ns=time.perf_counter_ns()-t\nt=time.perf_counter_ns(); result=analyze(raw,model); first_ns=time.perf_counter_ns()-t\nprint(json.dumps({"bundle_load_ns":load_ns,"first_prediction_ns":first_ns,"status":result.result.status,"score":result.result.score,"assessment":result.result.assessment,"model_version":result.result.model_version}))'
    start_ns = time.perf_counter_ns()
    proc = subprocess.run([sys.executable, '-c', child, str(ROOT), str(BUNDLE), str(path), str(duration)], capture_output=True, text=True, timeout=180, env={**os.environ, 'PYTHONPATH':str(ROOT / 'src')})
    process_to_result_ns = time.perf_counter_ns() - start_ns
    if proc.returncode:
        raise RuntimeError(proc.stderr[-2000:])
    return {'process_to_first_prediction_ns': process_to_result_ns, **json.loads(proc.stdout.strip().splitlines()[-1])}


FRESH_PROCESS_REPETITIONS = 10
report = {'schema':'r3-n2-bigru-latency-v1','created_utc':datetime.now(timezone.utc).isoformat(),'project_root':str(ROOT),'benchmark_kind':'fresh subprocess startup; not cold VM start','device':device_details(),'bundle':bundle_identity,'candidate_review_status':BUNDLE_REVIEW_STATUS,'deployment_ready':BUNDLE_DEPLOYMENT_READY,'configuration':{'batch_size':1,'recordings_per_inference_call':1,'model_chunk_batch_size':1,'warmup_calls':WARMUP_CALLS,'measured_calls':MEASURED_CALLS,'fresh_process_repetitions':FRESH_PROCESS_REPETITIONS,'durations_seconds': sorted(FIXTURES.keys()),'clock':'time.perf_counter_ns','tolerance':TOLERANCE},'groups':{},'errors':[]}
for seconds, path in FIXTURES.items():
    key = str(seconds)
    group = {'duration_seconds':seconds,'fixture_path':str(path),'warmup_raw_ns':[],'raw_samples_ns':[],'stage_samples_ns':{},'fresh_process_repetitions':[],'fresh_process_errors':[],'errors':[],'timeouts':0}
    report['groups'][key] = group
    if not Path(path).is_file():
        group['status']='missing_fixture'
        group['errors'].append('fixture path missing or not edited')
        print(f'MISSING {seconds}s fixture: {path}', flush=True)
        continue
    try:
        raw = prefix_bytes(path, seconds)
        group['actual_input_duration_seconds'] = seconds
        ref = analyze(raw, loaded)
        group['production_reference'] = {'status':ref.result.status,'score':ref.result.score,'assessment':ref.result.assessment,'model_version':ref.result.model_version}
        for repetition in range(FRESH_PROCESS_REPETITIONS):
            try:
                fresh = fresh_process(path, seconds)
                group['fresh_process_repetitions'].append(fresh)
            except subprocess.TimeoutExpired as exc:
                group['timeouts'] += 1
                group['fresh_process_errors'].append({'repetition':repetition+1,'type':'TimeoutExpired','message':str(exc)})
            except Exception as exc:
                group['fresh_process_errors'].append({'repetition':repetition+1,'type':type(exc).__name__,'message':str(exc)})
            print(f'{seconds}s fresh process {repetition+1}/{FRESH_PROCESS_REPETITIONS}', flush=True)
        fresh_metrics = {}
        for metric in ['process_to_first_prediction_ns','bundle_load_ns','first_prediction_ns']:
            vals = [int(item[metric]) for item in group['fresh_process_repetitions'] if item.get(metric) is not None]
            fresh_metrics[metric] = {'count':len(vals),'p50':percentile(vals,50),'p95':percentile(vals,95),'raw_samples_ns':vals}
        group['fresh_process_summary_ns'] = fresh_metrics
        for i in range(WARMUP_CALLS):
            _, stages, _ = instrumented_analyze(raw, loaded)
            group['warmup_raw_ns'].append(int(stages['total']))
            print(f'{seconds}s warmup {i+1}/{WARMUP_CALLS}', flush=True)
        parity_scores=[]
        parity_decisions=[]
        for i in range(MEASURED_CALLS):
            got, stages, calls = instrumented_analyze(raw, loaded)
            score_delta = abs(float(got.result.score)-float(ref.result.score)) if got.result.score is not None and ref.result.score is not None else None
            parity_scores.append(score_delta)
            parity_decisions.append(got.result.assessment == ref.result.assessment)
            group['raw_samples_ns'].append(int(stages['total']))
            for name, value in stages.items():
                group['stage_samples_ns'].setdefault(name,[]).append(int(value))
            if calls['feature_extraction'] < 1 or calls['bigru_prediction'] < 1:
                raise RuntimeError('feature extraction or Bi-GRU prediction wrapper was not invoked')
            print(f'{seconds}s measured {i+1}/{MEASURED_CALLS}', flush=True)
        group['parity']={'reference_match':bool(all(x is not None and x <= TOLERANCE for x in parity_scores)),'max_abs_score_delta':max((x for x in parity_scores if x is not None),default=None),'decision_flips':sum(not x for x in parity_decisions),'checks':len(parity_scores)}
        group['status']='completed' if len(group['fresh_process_repetitions']) == FRESH_PROCESS_REPETITIONS else 'completed_with_fresh_process_errors'
        group['summary_ns']={name:{'count':len(vals),'p50':percentile(vals,50),'p95':percentile(vals,95),'mean':float(np.mean(vals)) if vals else None} for name,vals in group['stage_samples_ns'].items()}
    except Exception as exc:
        group['status']='error'
        group['errors'].append(f'{type(exc).__name__}: {exc}')
        print(f'ERROR {seconds}s: {exc}',flush=True)

OUTPUT = ROOT/'reports/r3'
OUTPUT.mkdir(parents=True,exist_ok=True)
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REPORT_PATH=OUTPUT/f'r3_n2_bigru_latency_{stamp}.json'
REPORT_PATH.write_text(json.dumps(report,indent=2,ensure_ascii=False,default=str)+'\n',encoding='utf-8')
print('Saved report:',REPORT_PATH)
print('Status by duration:',{k:v['status'] for k,v in report['groups'].items()})


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

# Use the report generated by the benchmark
source = Path(REPORT_PATH)

with source.open("r", encoding="utf-8") as f:
    report = json.load(f)

summary = {
    "schema": "r3-n2-bigru-latency-summary-v1",
    "source_report": str(source),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "benchmark_kind": report.get("benchmark_kind"),
    "device": report.get("device"),
    "bundle": report.get("bundle"),
    "configuration": report.get("configuration"),
    "groups": {},
    "overall": {
        "duration_groups": len(report.get("groups", {})),
        "fresh_process_attempts": 0,
        "fresh_process_successes": 0,
        "fresh_process_failures": 0,
        "warm_measurements": 0,
        "parity_checks": 0,
        "parity_failures": 0
    },
    "top_level_errors": report.get("errors", [])
}

for duration, group in report.get("groups", {}).items():
    fresh = group.get("fresh_process_repetitions", [])
    fresh_errors = group.get("fresh_process_errors", [])
    parity = group.get("parity", {})
    warm = group.get("summary_ns", {})

    summary["groups"][duration] = {
        "status": group.get("status"),
        "fixture_path": group.get("fixture_path"),
        "actual_input_duration_seconds": group.get(
            "actual_input_duration_seconds"
        ),
        "warm_measurements": len(group.get("raw_samples_ns", [])),
        "warm_latency_ms": {
            stage: {
                "count": values.get("count"),
                "p50": (
                    values["p50"] / 1e6
                    if values.get("p50") is not None else None
                ),
                "p95": (
                    values["p95"] / 1e6
                    if values.get("p95") is not None else None
                ),
                "mean": (
                    values["mean"] / 1e6
                    if values.get("mean") is not None else None
                )
            }
            for stage, values in warm.items()
        },
        "parity": parity,
        "fresh_process": {
            "attempts": len(fresh) + len(fresh_errors),
            "successes": len(fresh),
            "failures": len(fresh_errors),
            "errors": fresh_errors,
            "latency_ms": {
                metric: {
                    "count": values.get("count"),
                    "p50": (
                        values["p50"] / 1e6
                        if values.get("p50") is not None else None
                    ),
                    "p95": (
                        values["p95"] / 1e6
                        if values.get("p95") is not None else None
                    )
                }
                for metric, values in group.get(
                    "fresh_process_summary_ns", {}
                ).items()
            }
        },
        "group_errors": group.get("errors", [])
    }

    summary["overall"]["fresh_process_attempts"] += (
        len(fresh) + len(fresh_errors)
    )
    summary["overall"]["fresh_process_successes"] += len(fresh)
    summary["overall"]["fresh_process_failures"] += len(fresh_errors)
    summary["overall"]["warm_measurements"] += len(
        group.get("raw_samples_ns", [])
    )
    summary["overall"]["parity_checks"] += parity.get("checks", 0)

    if not parity.get("reference_match", False):
        summary["overall"]["parity_failures"] += 1

# Save compact summary beside the original report
summary_path = source.with_name(
    source.stem + "_summary.json"
)

summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("SUMMARY FILE:", summary_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))
